In [0]:
from pyspark.sql import Row
from datetime import date

# Stock trading data - 10 days, 3 stocks
stock_data = [
    Row(stock_symbol="AAPL", trade_date=date(2024, 6, 1), volume=1000, closing_price=180.50),
    Row(stock_symbol="AAPL", trade_date=date(2024, 6, 2), volume=1500, closing_price=182.30),
    Row(stock_symbol="AAPL", trade_date=date(2024, 6, 3), volume=1200, closing_price=181.00),
    Row(stock_symbol="AAPL", trade_date=date(2024, 6, 4), volume=1800, closing_price=185.20),
    Row(stock_symbol="AAPL", trade_date=date(2024, 6, 5), volume=2000, closing_price=187.00),
    
    Row(stock_symbol="TSLA", trade_date=date(2024, 6, 1), volume=800, closing_price=250.00),
    Row(stock_symbol="TSLA", trade_date=date(2024, 6, 2), volume=900, closing_price=255.50),
    Row(stock_symbol="TSLA", trade_date=date(2024, 6, 3), volume=1100, closing_price=252.00),
    Row(stock_symbol="TSLA", trade_date=date(2024, 6, 4), volume=1300, closing_price=260.00),
    Row(stock_symbol="TSLA", trade_date=date(2024, 6, 5), volume=1500, closing_price=265.00),
    
    Row(stock_symbol="GOOGL", trade_date=date(2024, 6, 1), volume=600, closing_price=140.00),
    Row(stock_symbol="GOOGL", trade_date=date(2024, 6, 2), volume=700, closing_price=142.50),
    Row(stock_symbol="GOOGL", trade_date=date(2024, 6, 3), volume=650, closing_price=141.00),
    Row(stock_symbol="GOOGL", trade_date=date(2024, 6, 4), volume=750, closing_price=145.00),
    Row(stock_symbol="GOOGL", trade_date=date(2024, 6, 5), volume=800, closing_price=147.00),
]

stocks_df = spark.createDataFrame(stock_data)
display(stocks_df)

In [0]:
#running total of traded volume by stock
from pyspark.sql.window import Window
from pyspark.sql.functions import sum,avg, rank
window_spec = (
    Window.partitionBy("stock_symbol") #1.split data into groups
    .orderBy("trade_date") #2.sort within each group
    .rowsBetween(Window.unboundedPreceding , Window.currentRow) #Define window frame
)



In [0]:
stocks_with_cumulative = stocks_df.withColumn("cumulative_volume", sum("volume").over(window_spec))

In [0]:
#display
stocks_with_cumulative.orderBy("stock_symbol","trade_date").display()

In [0]:
#3-day moving avg stock prices

window_spec = (
    Window.partitionBy("stock_symbol")
        .orderBy("trade_date")
        .rowsBetween(-2, Window.currentRow)
)

stocks_with_moving_avg = stocks_df.withColumn("moving_avg_price", avg("closing_price").over(window_spec))

In [0]:
stocks_with_moving_avg.orderBy("stock_symbol","trade_date").display()

In [0]:
#top 2 trading volume days by stock
from pyspark.sql.functions import col

window_spec = (
    Window.partitionBy("stock_symbol")
    .orderBy(col("volume").desc())
)
    #rank() ranks everything in the partition.so no need to do .rowsBetween()

top_trading_days = (
    stocks_df.withColumn(
        "volume_rank", 
        rank().over(window_spec))
    .filter(col("volume_rank")<=2)
)


In [0]:
top_trading_days.orderBy("stock_symbol","volume_rank").display()